# EDA — Datos RoseAmor

**Autor:** Jairo Torres  
**Fecha:** 2026-03-03  

Primero voy a explorar los tres archivos CSV que me dieron para entender que tan limpios estan antes de armar el pipeline. La idea es revisar campo a campo y documentar lo que encuentre para despues decidir que hago con cada problema.

Archivos: `customers.csv`, `products.csv`, `orders.csv`

## Setup

In [33]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

RAW_DIR = os.path.join('..', 'data', 'raw')

CUSTOMERS_PATH = os.path.join(RAW_DIR, 'customers.csv')
PRODUCTS_PATH  = os.path.join(RAW_DIR, 'products.csv')
ORDERS_PATH    = os.path.join(RAW_DIR, 'orders.csv')

## Carga

Leo los archivos tal como estan, sin tocar nada todavia.

In [34]:
customers_raw = pd.read_csv(CUSTOMERS_PATH)
products_raw  = pd.read_csv(PRODUCTS_PATH)
orders_raw    = pd.read_csv(ORDERS_PATH)

print('Archivos cargados exitosamente.')
print(f'  customers : {customers_raw.shape[0]} filas, {customers_raw.shape[1]} columnas')
print(f'  products  : {products_raw.shape[0]} filas, {products_raw.shape[1]} columnas')
print(f'  orders    : {orders_raw.shape[0]} filas, {orders_raw.shape[1]} columnas')

Archivos cargados exitosamente.
  customers : 200 filas, 5 columnas
  products  : 60 filas, 4 columnas
  orders    : 1515 filas, 7 columnas


## Vistazo inicial — estructura y tipos

In [35]:
print('=== CUSTOMERS — Esquema ===')
print(customers_raw.dtypes)
print()
customers_raw.head()

=== CUSTOMERS — Esquema ===
customer_id    object
name           object
country        object
segment        object
created_at     object
dtype: object



,customer_id,name,country,segment,created_at
0,C0001,Diego Vargas,Ecuador,Wholesale,2024-06-17
1,C0002,Andrés Castro,France,Corporate,2023-12-01
2,C0003,Sofía Mendoza,UK,Distributor,2022-10-16
3,C0004,Carlos Torres,Germany,E-commerce,2024-05-24
4,C0005,Daniela Gómez,Germany,Distributor,2023-03-26


In [36]:
print('=== PRODUCTS — Esquema ===')
print(products_raw.dtypes)
print()
products_raw.head()

=== PRODUCTS — Esquema ===
sku          object
category     object
cost        float64
active         bool
dtype: object



,sku,category,cost,active
0,SKU0001,Gift Box,6.94,True
1,SKU0002,Seasonal,14.04,True
2,SKU0003,Seasonal,12.75,False
3,SKU0004,Seasonal,8.88,True
4,SKU0005,Standard,11.72,True


In [37]:
print('=== ORDERS — Esquema ===')
print(orders_raw.dtypes)
print()
orders_raw.head()

=== ORDERS — Esquema ===
order_id        object
customer_id     object
sku             object
quantity         int64
unit_price     float64
order_date      object
channel         object
dtype: object



,order_id,customer_id,sku,quantity,unit_price,order_date,channel
0,O000827,C0186,SKU0032,3,36.58,2025-05-28 00:00:00,wholesale
1,O001425,C0047,SKU0002,32,56.09,2025-02-08 00:00:00,ecommerce
2,O001056,C0117,SKU0010,48,60.25,2025-11-11 00:00:00,ecommerce
3,O000665,C0143,SKU0024,1,13.12,2024-05-24 00:00:00,retail
4,O001284,C0067,SKU0032,10,39.00,2026-01-14 00:00:00,ecommerce


In [38]:
# descripciones rapidas para ver rangos y detectar algo raro a primera vista
print('--- orders ---')
print(orders_raw.describe())
print()
print('--- products ---')
print(products_raw.describe())
# en customers no hay muchos numericos pero igual lo reviso
print()
print('--- customers ---')
print(customers_raw.describe(include='all'))

--- orders ---
       quantity  unit_price
count   1515.00     1505.00
mean      24.03       43.42
std       14.28       20.49
min       -9.00        8.05
25%       11.00       25.66
50%       24.00       42.87
75%       36.00       60.86
max       49.00       79.98

--- products ---
       cost
count 60.00
mean  13.78
std    7.29
min   -6.94
25%    9.21
50%   15.09
75%   18.75
max   24.14

--- customers ---
       customer_id          name  country      segment  created_at
count          200           200      195          195         200
unique         200           131       10            5         180
top          C0001  Diego Vargas  Germany  Distributor  2022-01-23
freq             1             3       27           41           2


---
## customers

### nulos

In [39]:
print('=== Valores nulos por campo ===')
null_counts = customers_raw.isnull().sum()
null_pct    = (null_counts / len(customers_raw) * 100).round(2)
pd.DataFrame({'nulos': null_counts, 'pct (%)': null_pct})

=== Valores nulos por campo ===


,nulos,pct (%)
customer_id,0,0.00
name,0,0.00
country,5,2.50
segment,5,2.50
created_at,0,0.00


In [40]:
# Registros con campo 'country' nulo
print(f"Registros sin 'country': {customers_raw['country'].isna().sum()}")
customers_raw[customers_raw['country'].isna()]

Registros sin 'country': 5


,customer_id,name,country,segment,created_at
26,C0027,Camila Ramírez,NaN,Corporate,2024-02-17
120,C0121,Fernando Gómez,NaN,Wholesale,2022-11-19
146,C0147,Ana Silva,NaN,Corporate,2023-01-22
153,C0154,Lucía Vargas,NaN,Distributor,2022-10-14
165,C0166,Andrés Torres,NaN,Distributor,2022-08-02


In [41]:
# Registros con campo 'segment' nulo
print(f"Registros sin 'segment': {customers_raw['segment'].isna().sum()}")
customers_raw[customers_raw['segment'].isna()]

Registros sin 'segment': 5


,customer_id,name,country,segment,created_at
77,C0078,Carlos Sánchez,Canada,NaN,2022-01-17
91,C0092,Camila Reyes,Mexico,NaN,2024-02-20
156,C0157,Daniela Gómez,USA,NaN,2023-10-25
164,C0165,Daniela Ortiz,Germany,NaN,2024-03-11
168,C0169,Ana Silva,Spain,NaN,2023-08-02


### duplicados en customer_id

In [42]:
dupes = customers_raw[customers_raw.duplicated(subset=['customer_id'], keep=False)]
print(f'Registros duplicados por customer_id: {len(dupes)}')
if len(dupes) > 0:
    print(dupes.sort_values('customer_id'))

Registros duplicados por customer_id: 0


### campo created_at
reviso si hay fechas invalidas o de futuro

In [43]:
parsed = pd.to_datetime(customers_raw['created_at'], errors='coerce')
invalid = customers_raw[parsed.isna()]
print(f"Fechas 'created_at' invalidas: {len(invalid)}")
if len(invalid) > 0:
    print(invalid[['customer_id', 'created_at']])

print(f"\nRango de fechas validas:")
print(f"  Min: {parsed.min()}")
print(f"  Max: {parsed.max()}")

Fechas 'created_at' invalidas: 0

Rango de fechas validas:
  Min: 2022-01-06 00:00:00
  Max: 2024-06-17 00:00:00


### distribucion de country y segment

In [44]:
print('=== Distribucion de country ===')
print(customers_raw['country'].value_counts(dropna=False))
print()
print('=== Distribucion de segment ===')
print(customers_raw['segment'].value_counts(dropna=False))

=== Distribucion de country ===
country
Germany        27
France         26
UK             23
USA            19
Ecuador        18
Mexico         18
Netherlands    18
Spain          18
Chile          15
Canada         13
NaN             5
Name: count, dtype: int64

=== Distribucion de segment ===
segment
Distributor    41
Retail         41
Wholesale      40
E-commerce     37
Corporate      36
NaN             5
Name: count, dtype: int64


---
## products

### nulos

In [45]:
null_counts = products_raw.isnull().sum()
null_pct    = (null_counts / len(products_raw) * 100).round(2)
pd.DataFrame({'nulos': null_counts, 'pct (%)': null_pct})

,nulos,pct (%)
sku,0,0.00
category,2,3.33
cost,0,0.00
active,0,0.00


In [46]:
# Productos sin categoria
print(f"Productos sin 'category': {products_raw['category'].isna().sum()}")
products_raw[products_raw['category'].isna()]

Productos sin 'category': 2


,sku,category,cost,active
35,SKU0036,NaN,-6.94,True
38,SKU0039,NaN,9.32,True


### campo cost
el describe mostro un min negativo, eso no tiene sentido — voy a ver cuantos son y cuales

In [47]:
neg_cost = products_raw[products_raw['cost'] < 0]
print(f'Productos con costo negativo: {len(neg_cost)}')
print()
print(neg_cost)
print()
print('Estadisticas del campo cost:')
print(products_raw['cost'].describe())

Productos con costo negativo: 3

        sku  category  cost  active
20  SKU0021    Luxury -6.94    True
33  SKU0034  Gift Box -6.94    True
35  SKU0036       NaN -6.94    True

Estadisticas del campo cost:
count   60.00
mean    13.78
std      7.29
min     -6.94
25%      9.21
50%     15.09
75%     18.75
max     24.14
Name: cost, dtype: float64


In [48]:
# los 3 valores negativos son -6.94 — coincide exactamente con el costo de SKU0001 (6.94)
# parece un error de signo en la fuente, no un valor inventado
# verifico que el valor absoluto cae dentro del rango normal
print('rango de costos positivos:')
print(products_raw[products_raw['cost'] > 0]['cost'].describe())
print()

print('valores abs de los negativos:', neg_cost['cost'].abs().tolist())# si, estan dentro del rango -> los voy a convertir con abs() en el pipeline

rango de costos positivos:
count   57.00
mean    14.87
std      5.63
min      5.77
25%     10.88
50%     15.24
75%     19.24
max     24.14
Name: cost, dtype: float64

valores abs de los negativos: [6.94, 6.94, 6.94]


### duplicados en sku

In [49]:
dupes_sku = products_raw[products_raw.duplicated(subset=['sku'], keep=False)]
print(f'SKUs duplicados: {len(dupes_sku)}')

SKUs duplicados: 0


### active y category
cuantos productos estan inactivos? y cuantos no tienen categoria?

In [50]:
print('=== Distribucion de active ===')
print(products_raw['active'].value_counts(dropna=False))
print()
print('=== Distribucion de category ===')
print(products_raw['category'].value_counts(dropna=False))

=== Distribucion de active ===
active
True     49
False    11
Name: count, dtype: int64

=== Distribucion de category ===
category
Standard    17
Gift Box    14
Luxury      12
Seasonal    10
Premium      5
NaN          2
Name: count, dtype: int64


In [51]:
# guardo los inactivos para despues cruzar con orders y ver si los puedo eliminar o no
inactive_skus = products_raw[products_raw['active'] == False]['sku'].tolist()
print(f'SKUs inactivos ({len(inactive_skus)}): {inactive_skus}')

SKUs inactivos (11): ['SKU0003', 'SKU0008', 'SKU0018', 'SKU0025', 'SKU0029', 'SKU0031', 'SKU0037', 'SKU0041', 'SKU0045', 'SKU0047', 'SKU0051']


---
## orders

### nulos

In [52]:
null_counts = orders_raw.isnull().sum()
null_pct    = (null_counts / len(orders_raw) * 100).round(2)
pd.DataFrame({'nulos': null_counts, 'pct (%)': null_pct})

,nulos,pct (%)
order_id,0,0.00
customer_id,0,0.00
sku,0,0.00
quantity,0,0.00
unit_price,10,0.66
order_date,0,0.00
channel,0,0.00


In [53]:
# Ordenes con unit_price nulo
null_price = orders_raw[orders_raw['unit_price'].isna()]
print(f"Ordenes sin 'unit_price': {len(null_price)}")
print()
print(null_price[['order_id', 'customer_id', 'sku', 'quantity', 'unit_price', 'order_date']])

Ordenes sin 'unit_price': 10

     order_id customer_id      sku  quantity  unit_price           order_date
53    O000242       C0009  SKU0037         6         NaN  2026-01-20 00:00:00
86    O000874       C0106  SKU0046        25         NaN  2025-07-14 00:00:00
252   O000019       C0011  SKU0049        27         NaN  2024-04-07 00:00:00
565   O000491       C0099  SKU0057        46         NaN  2025-01-08 00:00:00
622   O001280       C0082  SKU0054        46         NaN  2024-03-13 00:00:00
629   O000135       C0170  SKU0044         2         NaN  2024-01-11 00:00:00
653   O000716       C0183  SKU0004        47         NaN  2024-05-15 00:00:00
980   O000669       C0014  SKU0008         1         NaN  2024-04-29 00:00:00
985   O000383       C0173  SKU0056        46         NaN  2026-01-03 00:00:00
1134  O000993       C0082  SKU0056        36         NaN  2024-02-13 00:00:00


### duplicados en order_id

In [54]:
dupes_orders = orders_raw[orders_raw.duplicated(subset=['order_id'], keep=False)]
print(f'order_id duplicados (todas las ocurrencias): {len(dupes_orders)}')
if len(dupes_orders) > 0:
    print(dupes_orders.sort_values('order_id'))

order_id duplicados (todas las ocurrencias): 30
     order_id customer_id      sku  quantity  unit_price           order_date    channel
628   O000076       C0072  SKU0002         6       68.28  2025-06-08 00:00:00  ecommerce
957   O000076       C0072  SKU0002         6       68.28  2025-06-08 00:00:00  ecommerce
47    O000092       C0091  SKU0037        46       73.89  2024-05-06 00:00:00     export
519   O000092       C0091  SKU0037        46       73.89  2024-05-06 00:00:00     export
118   O000102       C0065  SKU0056        49       11.20  2026-01-24 00:00:00     export
391   O000102       C0065  SKU0056        49       11.20  2026-01-24 00:00:00     export
27    O000182       C0194  SKU0011        41       64.93  2024-10-30 00:00:00  ecommerce
1429  O000182       C0194  SKU0011        41       64.93  2024-10-30 00:00:00  ecommerce
444   O000331       C0135  SKU0059        20       10.24  2025-06-24 00:00:00  ecommerce
202   O000331       C0135  SKU0059        20       10.24  2025

### fechas invalidas en order_date

In [55]:
parsed_dates = pd.to_datetime(orders_raw['order_date'], errors='coerce')
invalid_dates = orders_raw[parsed_dates.isna()]
print(f'fechas que no se pudieron parsear: {len(invalid_dates)}')
print()
print(invalid_dates[['order_id', 'customer_id', 'sku', 'order_date', 'channel']])
# mes 13 y dia 40 — imposibles, no hay forma de recuperar la fecha real
# los voy a eliminar
print()
print(f'rango de fechas validas: {parsed_dates.min()} — {parsed_dates.max()}')

fechas que no se pudieron parsear: 6

     order_id customer_id      sku  order_date    channel
289   O000021       C0092  SKU0016  2025-13-40     retail
304   O000309       C0014  SKU0056  2025-13-40     retail
537   O000965       C0095  SKU0020  2025-13-40  ecommerce
804   O000711       C0046  SKU0005  2025-13-40     retail
921   O001109       C0050  SKU0027  2025-13-40  ecommerce
1022  O000226       C0005  SKU0043  2025-13-40  ecommerce

rango de fechas validas: 2024-01-01 00:00:00 — 2026-01-31 00:00:00


### quantity negativa o cero
puede ser una devolucion pero sin tabla de returns no lo puedo tratar como venta — los elimino

In [56]:
neg_qty  = orders_raw[orders_raw['quantity'] < 0]
zero_qty = orders_raw[orders_raw['quantity'] == 0]

print(f'Ordenes con quantity < 0: {len(neg_qty)}')
print(neg_qty[['order_id', 'customer_id', 'sku', 'quantity', 'unit_price', 'order_date']])
print()
print(f'Ordenes con quantity = 0: {len(zero_qty)}')

print()
print('Estadisticas del campo quantity:')
print(orders_raw['quantity'].describe())

Ordenes con quantity < 0: 8
     order_id customer_id      sku  quantity  unit_price           order_date
163   O000872       C0176  SKU0038        -9       61.62  2025-03-21 00:00:00
284   O000071       C0160  SKU0051        -6       21.28  2024-12-08 00:00:00
461   O001345       C0180  SKU0034        -9        8.77  2024-02-19 00:00:00
600   O001029       C0053  SKU0040        -7       37.79  2025-05-08 00:00:00
1016  O000378       C0123  SKU0059        -5       34.69  2024-06-02 00:00:00
1264  O000891       C0102  SKU0025        -8       41.96  2024-06-26 00:00:00
1269  O001152       C0100  SKU0040        -2       27.65  2024-01-18 00:00:00
1401  O001422       C0195  SKU0049        -1       31.33  2025-06-25 00:00:00

Ordenes con quantity = 0: 0

Estadisticas del campo quantity:
count   1515.00
mean      24.03
std       14.28
min       -9.00
25%       11.00
50%       24.00
75%       36.00
max       49.00
Name: quantity, dtype: float64


### channel — consistencia de formato

In [57]:
print('channel valores unicos (raw):')
print(orders_raw['channel'].value_counts(dropna=False))
print()
# normalizo para ver si hay variantes
normalized = orders_raw['channel'].str.lower().str.strip()
print('despues de lower + strip:')
print(normalized.value_counts(dropna=False))
# en este caso no hay diferencia pero lo dejo en el pipeline por si los nuevos archivos traen variaciones

channel valores unicos (raw):
channel
ecommerce    550
retail       386
wholesale    369
export       210
Name: count, dtype: int64

despues de lower + strip:
channel
ecommerce    550
retail       386
wholesale    369
export       210
Name: count, dtype: int64


### unit_price nulo — como imputar?
no quiero eliminar la orden entera por falta de precio, voy a ver si el SKU tiene historial

In [58]:
# Para las ordenes con unit_price nulo, verificar si el SKU tiene historial de precios
null_price_orders = orders_raw[orders_raw['unit_price'].isna()][['order_id', 'sku']]
print('SKUs afectados por unit_price nulo:')
print(null_price_orders)
print()

# Precio promedio disponible por SKU en el resto del dataset
for sku in null_price_orders['sku'].unique():
    other_prices = orders_raw[(orders_raw['sku'] == sku) & (orders_raw['unit_price'].notna())]['unit_price']
    if len(other_prices) > 0:
        print(f'  SKU {sku} -> precio promedio historico: {other_prices.mean():.2f} (n={len(other_prices)})')
    else:
        print(f'  SKU {sku} -> sin historial de precios. Se usara el promedio global.')

print(f'\nPrecio promedio global: {orders_raw["unit_price"].mean():.2f}')

SKUs afectados por unit_price nulo:
     order_id      sku
53    O000242  SKU0037
86    O000874  SKU0046
252   O000019  SKU0049
565   O000491  SKU0057
622   O001280  SKU0054
629   O000135  SKU0044
653   O000716  SKU0004
980   O000669  SKU0008
985   O000383  SKU0056
1134  O000993  SKU0056

  SKU SKU0037 -> precio promedio historico: 44.44 (n=26)
  SKU SKU0046 -> precio promedio historico: 43.56 (n=35)
  SKU SKU0049 -> precio promedio historico: 47.02 (n=23)
  SKU SKU0057 -> precio promedio historico: 43.05 (n=23)
  SKU SKU0054 -> precio promedio historico: 38.36 (n=26)
  SKU SKU0044 -> precio promedio historico: 40.13 (n=22)
  SKU SKU0004 -> precio promedio historico: 49.25 (n=22)
  SKU SKU0008 -> precio promedio historico: 38.99 (n=17)
  SKU SKU0056 -> precio promedio historico: 41.26 (n=31)

Precio promedio global: 43.42


---
## integridad referencial

cruce entre tablas — quiero asegurarme de que no haya ordenes con clientes o productos que no existen

In [59]:
# Claves validas en tablas de dimension
valid_customers = set(customers_raw['customer_id'].dropna())
valid_skus      = set(products_raw['sku'].dropna())

# Ordenes con customer_id que no existe en customers
orphan_customers = orders_raw[~orders_raw['customer_id'].isin(valid_customers)]
print(f'Ordenes con customer_id inexistente en customers: {len(orphan_customers)}')
if len(orphan_customers) > 0:
    print(orphan_customers[['order_id', 'customer_id', 'sku']].head(10))

Ordenes con customer_id inexistente en customers: 0


In [60]:
# Ordenes con sku que no existe en products
orphan_skus = orders_raw[~orders_raw['sku'].isin(valid_skus)]
print(f'Ordenes con sku inexistente en products: {len(orphan_skus)}')
if len(orphan_skus) > 0:
    print(orphan_skus[['order_id', 'customer_id', 'sku']].head(10))

Ordenes con sku inexistente en products: 0


In [61]:
# tambien verifico si los productos inactivos tienen ordenes — si tienen no los puedo borrar
orders_with_inactive = orders_raw[orders_raw['sku'].isin(inactive_skus)]
print(f'ordenes con SKU inactivo: {len(orders_with_inactive)}')
print()
print(orders_with_inactive['sku'].value_counts())
# tienen historial -> los mantengo en la BD y los filtro solo en la UI

ordenes con SKU inactivo: 274

sku
SKU0018    34
SKU0003    30
SKU0051    29
SKU0037    27
SKU0031    27
SKU0029    24
SKU0045    23
SKU0025    22
SKU0047    21
SKU0041    19
SKU0008    18
Name: count, dtype: int64


---
## resumen de todo lo que encontre

In [62]:
parsed_dates_final = pd.to_datetime(orders_raw['order_date'], errors='coerce')

hallazgos = [
    ('customers', 'country',     'nulos',              int(customers_raw['country'].isna().sum()),    'fillna Unknown'),
    ('customers', 'segment',     'nulos',              int(customers_raw['segment'].isna().sum()),    'fillna Unknown'),
    ('customers', 'customer_id', 'duplicados',         int(customers_raw.duplicated(subset=['customer_id']).sum()), 'keep first'),
    ('products',  'category',    'nulos',              int(products_raw['category'].isna().sum()),    'fillna Uncategorized'),
    ('products',  'cost',        'negativos',          int((products_raw['cost'] < 0).sum()),         'abs()'),
    ('products',  'sku',         'duplicados',         int(products_raw.duplicated(subset=['sku']).sum()), 'keep first'),
    ('orders',    'unit_price',  'nulos',              int(orders_raw['unit_price'].isna().sum()),    'promedio del SKU'),
    ('orders',    'quantity',    'negativos/cero',     int((orders_raw['quantity'] <= 0).sum()),      'eliminar'),
    ('orders',    'order_date',  'fechas invalidas',   int(parsed_dates_final.isna().sum()),         'eliminar'),
    ('orders',    'order_id',    'duplicados',         int(orders_raw.duplicated(subset=['order_id']).sum()), 'keep first'),
    ('orders',    'channel',     'formato',            0,                                             'lower + strip'),
    ('orders',    'customer_id', 'FK sin match',       len(orphan_customers),                        'revisar'),
    ('orders',    'sku',         'FK sin match',       len(orphan_skus),                             'revisar'),
]

df_hallazgos = pd.DataFrame(hallazgos, columns=['tabla', 'campo', 'problema', 'filas_afectadas', 'decision'])
df_hallazgos

,tabla,campo,problema,filas_afectadas,decision
0,customers,country,nulos,5,fillna Unknown
1,customers,segment,nulos,5,fillna Unknown
2,customers,customer_id,duplicados,0,keep first
3,products,category,nulos,2,fillna Uncategorized
4,products,cost,negativos,3,abs()
5,products,sku,duplicados,0,keep first
6,orders,unit_price,nulos,10,promedio del SKU
7,orders,quantity,negativos/cero,8,eliminar
8,orders,order_date,fechas invalidas,6,eliminar
9,orders,order_id,duplicados,15,keep first


---
## cuanto se pierde con los eliminados?

In [63]:
# Validacion rapida: resumen del impacto de las decisiones de eliminacion
print('=== Impacto estimado de registros a ELIMINAR ===')

p_dates_mask = pd.to_datetime(orders_raw['order_date'], errors='coerce').isna()
neg_qty_mask = orders_raw['quantity'] <= 0
dupes_mask   = orders_raw.duplicated(subset=['order_id'], keep='first')

# Las mascaras son independientes; calculamos union para no sobrecontar
to_drop = p_dates_mask | neg_qty_mask | dupes_mask

print(f'  Fechas invalidas       : {p_dates_mask.sum()} filas')
print(f'  Quantities invalidas   : {neg_qty_mask.sum()} filas')
print(f'  Duplicados order_id    : {dupes_mask.sum()} filas')
print(f'  Total a eliminar (union): {to_drop.sum()} filas')
print(f'  Filas originales       : {len(orders_raw)}')
print(f'  Filas resultantes      : {len(orders_raw) - to_drop.sum()}')
print(f'  Retencion              : {((len(orders_raw) - to_drop.sum()) / len(orders_raw) * 100):.1f}%')

=== Impacto estimado de registros a ELIMINAR ===
  Fechas invalidas       : 6 filas
  Quantities invalidas   : 8 filas
  Duplicados order_id    : 15 filas
  Total a eliminar (union): 29 filas
  Filas originales       : 1515
  Filas resultantes      : 1486
  Retencion              : 98.1%


con esto ya tengo claro que hay que hacer en cada campo — lo implemento en el script `etl/etl_pipeline.py`